# 02 — Strategy Comparison

This notebook runs all 5 intraday momentum strategies on SPY data and compares
their performance side-by-side. Each strategy is a variation of the same core idea
(sigma-band breakouts with volatility-targeted sizing) but differs in entry/exit
logic.

| Strategy | Key Difference |
|----------|---------------|
| 0 | Baseline — VWAP exit, 30-min checks |
| 1 | Asymmetric intervals (30m entry, 5m exit) |
| 2 | EMA trend filter for entry confirmation |
| 3 | Exit confirmation counter (4 bars) |
| 4 | EMA + exit confirmation (most conservative) |

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from intraday_momentum.data.provider import DataProvider
from intraday_momentum.strategies import Strategy0, Strategy1, Strategy2, Strategy3, Strategy4
from intraday_momentum.backtest.engine import BacktestEngine

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

print("Imports OK.")

## 1. Load Data

We fetch daily data for volatility estimation and minute data for signal generation.
Note: yfinance provides max ~60 days of 1-minute data.

In [ ]:
provider = DataProvider("SPY", cache_dir="../data_cache")

daily = provider.get_daily_data("2025-01-01", "2026-05-01")
minute = provider.get_data("2026-04-15", "2026-05-04", interval="2m")

print(f"Daily:  {len(daily)} bars ({daily.index[0].date()} → {daily.index[-1].date()})")
print(f"Minute: {len(minute)} bars ({minute.index[0]} → {minute.index[-1]})")

## 2. Initialize Strategies

All strategies share the same base parameters (lookback=14, vol_target=2%)
so the comparison is fair — only the entry/exit logic differs.

In [ ]:
strategies = {
    "S0 — Baseline":        Strategy0(lookback=14, vol_target=0.02),
    "S1 — Asymmetric":      Strategy1(lookback=14, vol_target=0.02, entry_interval=30, exit_interval=5),
    "S2 — EMA Filter":      Strategy2(lookback=14, vol_target=0.02, ema_period=100),
    "S3 — Exit Confirm":    Strategy3(lookback=14, vol_target=0.02, exit_confirmation_bars=4),
    "S4 — EMA + Confirm":   Strategy4(lookback=14, vol_target=0.02, ema_period=100, exit_confirmation_bars=4),
}

engine = BacktestEngine(initial_capital=100_000, commission_per_share=0.005)

print(f"Initialized {len(strategies)} strategies with BacktestEngine")
print(f"Initial capital: $100,000 | Commission: $0.005/share")

## 3. Run Backtests

Execute each strategy on the same data and collect results.

In [ ]:
results = {}

for name, strategy in strategies.items():
    print(f"Running {name}...", end=" ")
    result = engine.run(strategy, daily, minute)
    results[name] = result
    print(f"Done — {result.num_trades} trades, "
          f"return: {result.total_return:+.2f}%")

print(f"\nAll strategies completed.")

## 4. Performance Summary Table

Side-by-side comparison of key metrics for all strategies.

In [ ]:
rows = []
for name, result in results.items():
    s = result.summary()
    rows.append({
        "Strategy": name,
        "Final Equity ($)": f"{s['final_equity']:,.0f}",
        "Return (%)": f"{s['total_return_pct']:+.2f}",
        "# Trades": int(s["num_trades"]),
        "Win Rate (%)": f"{s['win_rate'] * 100:.1f}",
        "Avg Win (%)": f"{s['avg_win_pct']:.2f}",
        "Avg Loss (%)": f"{s['avg_loss_pct']:.2f}",
        "Sharpe": f"{s['sharpe_ratio']:.2f}",
        "Max DD (%)": f"{s['max_drawdown_pct']:.2f}",
    })

summary_df = pd.DataFrame(rows).set_index("Strategy")
summary_df

## 5. Equity Curves

Overlay of all strategy equity curves to visualize relative performance over time.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

for (name, result), color in zip(results.items(), colors):
    if not result.equity_curve.empty:
        curve = result.equity_curve.sort_index()
        ax.plot(curve.index, curve.values, label=name, color=color, linewidth=1.2)

ax.axhline(y=100_000, color="gray", linestyle="--", alpha=0.5, label="Starting capital")
ax.set_ylabel("Portfolio Value ($)")
ax.set_xlabel("Date")
ax.set_title("Strategy Comparison — Equity Curves")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Drawdown Comparison

Underwater plot showing the drawdown from peak equity for each strategy.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for (name, result), color in zip(results.items(), colors):
    if not result.equity_curve.empty:
        curve = result.equity_curve.sort_index()
        peak = curve.expanding().max()
        drawdown = (curve - peak) / peak * 100
        ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.15, color=color)
        ax.plot(drawdown.index, drawdown.values, label=name, color=color, linewidth=0.8)

ax.set_ylabel("Drawdown (%)")
ax.set_xlabel("Date")
ax.set_title("Strategy Comparison — Drawdowns")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Trade Return Distributions

Histograms of per-trade returns for each strategy. This reveals the
risk/reward profile beyond aggregate metrics.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)

for ax, (name, result), color in zip(axes, results.items(), colors):
    if result.trades:
        trade_returns = [t.return_pct for t in result.trades]
        ax.hist(trade_returns, bins=30, color=color, alpha=0.7, edgecolor="white")
    ax.set_title(name.split("—")[0].strip(), fontsize=10)
    ax.set_xlabel("Return (%)")
    ax.axvline(x=0, color="black", linewidth=0.5)

axes[0].set_ylabel("Count")
fig.suptitle("Per-Trade Return Distributions", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Trade Statistics

Detailed breakdown of trade characteristics per strategy.

In [ ]:
for name, result in results.items():
    if not result.trades:
        print(f"{name}: No trades")
        continue

    trade_returns = [t.return_pct for t in result.trades]
    longs = [t for t in result.trades if t.direction == 1]
    shorts = [t for t in result.trades if t.direction == -1]

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  Total trades:  {len(result.trades)}")
    print(f"  Long / Short:  {len(longs)} / {len(shorts)}")
    print(f"  Win rate:      {result.win_rate:.1%}")
    print(f"  Avg return:    {np.mean(trade_returns):.3f}%")
    print(f"  Std return:    {np.std(trade_returns):.3f}%")
    print(f"  Best trade:    {max(trade_returns):.2f}%")
    print(f"  Worst trade:   {min(trade_returns):.2f}%")

## 9. Cumulative Trades Over Time

Shows the pacing of trade activity for each strategy.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for (name, result), color in zip(results.items(), colors):
    if result.trades:
        trade_times = [t.exit_time for t in result.trades]
        ax.plot(trade_times, range(1, len(trade_times) + 1),
                label=name, color=color, linewidth=1.2)

ax.set_ylabel("Cumulative Trades")
ax.set_xlabel("Date")
ax.set_title("Trade Activity Over Time")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusions

Key takeaways from the strategy comparison:

1. **Baseline (S0)** provides the most trading opportunities but may be prone to
   whipsaw exits.
2. **Asymmetric intervals (S1)** exits faster, reducing exposure time per trade.
3. **EMA filter (S2)** is more selective on entries, trading less frequently but
   potentially with higher conviction.
4. **Exit confirmation (S3)** holds positions longer, requiring sustained adverse
   moves before exiting — reduces whipsaw cost.
5. **Combined (S4)** is the most conservative variant, filtering both entries and exits.

The best strategy depends on the metric prioritized:
- **Highest Sharpe** → less frequent trading, more filters
- **Highest total return** → more aggressive entry logic
- **Lowest drawdown** → exit confirmation + trend filters

Next steps: optimize parameters using CMA-ES (see optimization module).